# Exp 1 — Signal Smoothness on the Music Similarity Graph

<a href="https://colab.research.google.com/github/zixian0821-zoe/EN553744-final-project/blob/main/exp1_signal_smoothness/01_smoothness.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Course:** EN.553.744 Data Science for Large-Scale Graphs (Spring 2026, JHU AMS)
**Team:** Yunwei Chai · Yang Song · Zixian Zhou

## What this notebook shows

Each user's top-20 book genre distribution is treated as a graph signal $Y \in \mathbb{R}^{N \times 20}$ on a music similarity k-NN graph $G_{\text{src}}$. We compare its Dirichlet energy

$$E(Y; L) = \frac{1}{20}\sum_{d=1}^{20} y_d^\top L\, y_d \quad (\text{each } y_d \text{ z-scored})$$

to a null distribution from 200 random node-ID permutations of the same adjacency. Under $H_0$ (book signal independent of music topology) the permutation null and the observed energy should be indistinguishable.

**Headline.** $E_{\text{music}} \ll E_{\text{perm}}$, $z \approx -10$, $p < 0.005$ (0/200 permutations below observed). The book signal is statistically smoother on the music graph than on a random graph with identical edge weights — direct evidence that the music topology is informative for book preferences, justifying the GSP framing of Exp 2.

## Why this notebook only loads results (no re-run)

The 200-permutation test on $N \approx 16\text{k}$ users takes ~10 minutes. The results are deterministic given `np.random.seed(42)`, so we save the output JSONs to Drive and just load + print them here. The full re-run code lives in `total_variation_colab.py` and `rerun_on_exp2_graph.py` in the same folder.

## 0. Setup — point at the results folder

In [ ]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Folder you shared:
    # https://drive.google.com/drive/folders/1Sya4GcmaKJqM0EVqgpxeYmdgqmU9D80s
    # Add it to your own Drive (right-click -> Add shortcut to Drive) and put the
    # path below.  Or change to wherever you actually keep the JSONs.
    RESULTS_DIR = '/content/drive/MyDrive/EN553744_final_project/results/exp1_signal_smoothness'
else:
    RESULTS_DIR = os.path.abspath('../../drive_upload/results/exp1_signal_smoothness')

print('IN_COLAB    :', IN_COLAB)
print('RESULTS_DIR :', RESULTS_DIR)
print('exists      :', os.path.isdir(RESULTS_DIR))
if os.path.isdir(RESULTS_DIR):
    print('contents    :', sorted(os.listdir(RESULTS_DIR)))

## 1. Load both result files

- `permutation_test_k10.json` — fresh top-20 k-NN music graph, $k=10$ cosine, 200 perms
- `permutation_test_exp2_graph.json` — re-run on the **actual** music adjacency Exp 2's models train on (top-30 L1 music profile, $k=10$ cosine), 200 perms

Both should agree qualitatively; the second one is the version cited in the report.

In [ ]:
def load_json(path):
    if not os.path.isfile(path):
        print(f'[missing] {path}')
        return None
    with open(path) as f:
        d = json.load(f)
    print(f'[loaded ] {os.path.basename(path)}  ({len(json.dumps(d))} chars)')
    return d

res_k10  = load_json(os.path.join(RESULTS_DIR, 'permutation_test_k10.json'))
res_exp2 = load_json(os.path.join(RESULTS_DIR, 'permutation_test_exp2_graph.json'))

## 2. Headline numbers — fresh top-20 k-NN graph

In [ ]:
if res_k10 is None:
    print('[skip] permutation_test_k10.json not loaded')
else:
    r = res_k10
    print('=' * 65)
    print('Exp 1 — fresh music k-NN graph (top-20 profile, k=10 cosine)')
    print('=' * 65)
    print(f"Users                        : {r['n_users']}")
    print(f"Edges                        : {r['n_edges']}  (avg deg {r['avg_degree']:.1f})")
    print(f"Permutations                 : {r['n_permutations']}")
    print()
    print(f"Music graph energy           : {r['music_graph_energy']:>10.2f}")
    print(f"Random mean +/- std          : {r['random_mean']:>10.2f} +/- {r['random_std']:.2f}")
    print(f"Random range                 : [{r['random_min']:.2f}, {r['random_max']:.2f}]")
    print(f"z-score                      : {r['z_score']:>10.2f}")
    print(f"p-value                      : {r['p_value']}")
    print(f"Energy ratio (music/random)  : {r['energy_ratio']:>10.4f}")
    print(f"Smoothness improvement       : {r['smoothness_percent']:>10.1f}%")
    print()
    print(f"Per-genre  smoother / similar / rougher  : "
          f"{r['per_genre_smoother']} / {r['per_genre_similar']} / {r['per_genre_rougher']}  "
          f"(of {r['per_genre_total']})")

## 3. Headline numbers — Exp 2 source graph (the version cited in the report)

In [ ]:
if res_exp2 is None:
    print('[skip] permutation_test_exp2_graph.json not loaded')
else:
    r = res_exp2
    print('=' * 65)
    print('Exp 1 — Exp 2 source graph (top-30 L1 music profile, k=10 cosine)')
    print('=' * 65)
    print(f"Users                        : {r['n_users']}")
    print(f"Edges (sym nnz / undirected) : {r['n_edges_symmetric_nnz']} / {r['n_edges_undirected']}")
    print(f"Avg degree (directed entries): {r['avg_degree_directed_entries']:.2f}")
    print(f"Permutations                 : {r['n_permutations']}")
    print()
    print(f"Music graph energy           : {r['music_graph_energy']:>10.2f}")
    print(f"Random mean +/- std          : {r['random_mean']:>10.2f} +/- {r['random_std']:.2f}")
    print(f"Random range                 : [{r['random_min']:.2f}, {r['random_max']:.2f}]")
    print(f"z-score                      : {r['z_score']:>10.2f}")
    print(f"Music energy below {r['n_below_permutations']:>3}/{r['n_permutations']} permutations")
    print(f"p-value (lower bound)        : <= {r['p_value_lower_bound']:.4f}")
    print(f"Energy ratio (music/random)  : {r['energy_ratio']:>10.4f}")
    print(f"Smoothness improvement       : {r['smoothness_percent']:>10.2f}%")
    print()
    print('Per-genre breakdown')
    print(f"  ratio < 0.95              : {r['per_genre_smoother_by_ratio_lt_0.95']:>3} / {r['per_genre_total']}")
    print(f"  z < -2                    : {r['per_genre_smoother_by_z_lt_neg2']:>3} / {r['per_genre_total']}")
    print(f"  z < -3 (significant)      : {r['per_genre_smoother_by_z_lt_neg3']:>3} / {r['per_genre_total']}")
    print(f"  rougher (z > 2 & r > 1.05): {r['per_genre_rougher_significant']:>3} / {r['per_genre_total']}")

## 4. Per-genre table

Reconstructed from the `per_genre_ratios` and `per_genre_zscores` arrays saved in the JSON.

In [ ]:
if res_exp2 is None:
    print('[skip] need permutation_test_exp2_graph.json for per-genre table')
else:
    ratios  = np.asarray(res_exp2['per_genre_ratios'])
    zs      = np.asarray(res_exp2['per_genre_zscores'])
    n_dim   = len(ratios)

    print('PER-GENRE BREAKDOWN  (Exp 2 source graph)')
    print('-' * 50)
    print(f"{'Genre':<8} {'Ratio':>8} {'z-score':>9}  Status")
    print('-' * 50)
    for d in range(n_dim):
        if   ratios[d] < 0.95: status = '[smoother]'
        elif ratios[d] > 1.05: status = '[rougher]'
        else:                  status = '[~same]'
        print(f'G{d:<6} {ratios[d]:>8.3f} {zs[d]:>9.2f}  {status}')

## 5. Plots

Three panels:

1. **Permutation null + observed.** The raw 200 perm energies aren't saved in the JSON, so panel 1 draws a Gaussian PDF using the saved mean/std as a visual proxy (clearly labeled). The vertical line is the observed music-graph energy.
2. **Per-genre energy ratio** — reconstructed exactly from the JSON.
3. **Per-genre z-score** — reconstructed exactly from the JSON.

In [ ]:
if res_exp2 is None:
    print('[skip] need permutation_test_exp2_graph.json for plots')
else:
    r       = res_exp2
    ratios  = np.asarray(r['per_genre_ratios'])
    zs      = np.asarray(r['per_genre_zscores'])
    n_dim   = len(ratios)
    me      = r['music_graph_energy']
    mu, sd  = r['random_mean'], r['random_std']
    rmin    = r['random_min']
    rmax    = r['random_max']
    zscore  = r['z_score']
    p_lb    = r['p_value_lower_bound']

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Panel 1 — Gaussian approximation of permutation null + observed
    ax = axes[0]
    xs = np.linspace(min(me, mu - 4 * sd), mu + 4 * sd, 400)
    pdf = np.exp(-0.5 * ((xs - mu) / sd) ** 2) / (sd * np.sqrt(2 * np.pi))
    ax.fill_between(xs, pdf, color='lightcoral', alpha=0.6,
                    label=f'Gaussian approx (mu={mu:.1f}, sd={sd:.1f})')
    ax.axvspan(rmin, rmax, color='lightcoral', alpha=0.25,
               label=f'Observed perm range [{rmin:.0f}, {rmax:.0f}]')
    ax.axvline(me, color='steelblue', linewidth=2.5, linestyle='--',
               label=f'Music graph (E={me:.1f})')
    ax.set_xlabel('Average Dirichlet Energy', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title(f'Perm null (Gaussian approx)  '
                 f'p<={p_lb:.4f},  z={zscore:.2f}', fontsize=12)
    ax.legend(fontsize=9, loc='upper left')

    # Panel 2 — per-genre ratio
    ax     = axes[1]
    colors = ['steelblue' if x < 0.95 else 'lightcoral' if x > 1.05 else 'gray'
              for x in ratios]
    ax.bar(range(n_dim), ratios, color=colors, edgecolor='black', linewidth=0.3)
    ax.axhline(1.00, color='black',      linewidth=1.0, linestyle='--')
    ax.axhline(0.95, color='steelblue',  linewidth=0.8, linestyle=':', alpha=0.7)
    ax.axhline(1.05, color='lightcoral', linewidth=0.8, linestyle=':', alpha=0.7)
    ax.set_xlabel('Genre Index', fontsize=12)
    ax.set_ylabel('Energy Ratio (Music / Random)', fontsize=12)
    ax.set_title('Per-Genre Smoothness', fontsize=12)
    ax.set_ylim(0.85, 1.10)

    # Panel 3 — per-genre z
    ax       = axes[2]
    colors_z = ['steelblue' if z < -2 else 'lightcoral' if z > 2 else 'gray'
                for z in zs]
    ax.bar(range(n_dim), zs, color=colors_z, edgecolor='black', linewidth=0.3)
    ax.axhline( 0, color='black',      linewidth=1.0)
    ax.axhline(-2, color='steelblue',  linewidth=0.8, linestyle=':', alpha=0.7,
               label='z = +/- 2')
    ax.axhline( 2, color='lightcoral', linewidth=0.8, linestyle=':', alpha=0.7)
    ax.set_xlabel('Genre Index', fontsize=12)
    ax.set_ylabel('z-score (negative = smoother)', fontsize=12)
    ax.set_title('Per-Genre Statistical Significance', fontsize=12)
    ax.legend(fontsize=10)

    plt.suptitle(f'Exp 1 — Dirichlet Energy on Music Graph  '
                 f'(N={r["n_users"]}, perms={r["n_permutations"]})',
                 fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## 6. Side-by-side: fresh k-NN vs Exp 2 source graph

In [ ]:
if res_k10 and res_exp2:
    print('=' * 70)
    print('SIDE-BY-SIDE COMPARISON  —  same signal, two graph constructions')
    print('=' * 70)
    print(f"{'metric':<32} {'fresh k-NN (k10)':>18} {'Exp 2 source graph':>20}")
    print('-' * 70)
    print(f"{'music graph energy':<32} {res_k10['music_graph_energy']:>18.2f} "
          f"{res_exp2['music_graph_energy']:>20.2f}")
    print(f"{'random mean':<32} {res_k10['random_mean']:>18.2f} "
          f"{res_exp2['random_mean']:>20.2f}")
    print(f"{'random std':<32} {res_k10['random_std']:>18.2f} "
          f"{res_exp2['random_std']:>20.2f}")
    print(f"{'z-score':<32} {res_k10['z_score']:>18.2f} "
          f"{res_exp2['z_score']:>20.2f}")
    print(f"{'energy ratio':<32} {res_k10['energy_ratio']:>18.4f} "
          f"{res_exp2['energy_ratio']:>20.4f}")
    print(f"{'smoothness improvement (%)':<32} {res_k10['smoothness_percent']:>18.2f} "
          f"{res_exp2['smoothness_percent']:>20.2f}")
    print()
    print('Both graph constructions yield the same qualitative conclusion:')
    print('the book signal is significantly smoother on the music topology than on')
    print('a random graph with identical edge weights — robust to graph construction.')

## 7. Summary block (paste into report)

In [ ]:
if res_exp2:
    r = res_exp2
    p_str = f"<= {r['p_value_lower_bound']:.4f}"
    print('=' * 65)
    print('SUMMARY FOR REPORT  (Exp 1, Exp 2 source graph)')
    print('=' * 65)
    print(f"""
Experiment      : Dirichlet Energy Permutation Test
  Dataset       : Amazon Reviews 2023, CDs & Vinyl  ->  Books
  Users         : {r['n_users']}
  Graph         : k-NN (k=10), cosine similarity on top-30 L1 music profile
                  (the actual adjacency Exp 2 trains on)
  Signal        : 20-dim book genre distribution (L1-normalized per user)
  Permutations  : {r['n_permutations']} (node-ID shuffle on signal, identical edge weights)
  Seed          : 42

Results
  Music graph energy   : {r['music_graph_energy']:.2f}
  Random mean +/- std  : {r['random_mean']:.2f} +/- {r['random_std']:.2f}
  Energy ratio         : {r['energy_ratio']:.4f}   ({r['smoothness_percent']:.1f}% smoother)
  z-score              : {r['z_score']:.2f}
  p-value              : {p_str}   ({r['n_below_permutations']}/{r['n_permutations']} perms below observed)

Per-genre breakdown
  Significant smoother (z < -3) : {r['per_genre_smoother_by_z_lt_neg3']:>2} / {r['per_genre_total']}
  Smoother by trend  (z < -2)   : {r['per_genre_smoother_by_z_lt_neg2']:>2} / {r['per_genre_total']}
  Significant rougher           : {r['per_genre_rougher_significant']:>2} / {r['per_genre_total']}

Conclusion
  The book genre signal has significantly lower Dirichlet energy on the
  music similarity graph than on random permutations of the same graph.
  Users with similar music preferences also tend to have similar book
  preferences, justifying the use of GSP / GCN methods for cross-domain
  prediction in Exp 2 onwards.
""")